# CLM-0.3 — Public Release Benchmark

Public-release evidence runner: historical Transformer↔TextNCA foundation, a new same-checkpoint TextNCA↔CLM machinery bridge, formal CLM-0.3d developmental evidence, and publication-ready figures. Enable two T4 GPUs for the preferred run.

In [ ]:
import subprocess, sys
from pathlib import Path

ROOT = Path('/kaggle/working/mini-cells')
BRANCH = 'codex/clm-0.3-release-benchmark'
REPO = 'https://github.com/ArcheLabs/mini-cells.git'

if not (ROOT / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO, str(ROOT)], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=ROOT, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=ROOT, check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=ROOT, check=True)

HEAD = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=ROOT, text=True).strip()
TREE = subprocess.check_output(['git', 'rev-parse', 'HEAD^{tree}'], cwd=ROOT, text=True).strip()
DIRTY = subprocess.check_output(['git', 'status', '--porcelain', '--untracked-files=no'], cwd=ROOT, text=True).strip()
print({'HEAD': HEAD, 'tree': TREE, 'tracked_dirty': bool(DIRTY)})
assert not DIRTY, 'Formal release benchmark requires a clean tracked Git tree'


In [ ]:
import torch
print({
    'python': sys.version.split()[0],
    'torch': torch.__version__,
    'cuda': torch.version.cuda,
    'gpu_count': torch.cuda.device_count(),
    'gpus': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
})
assert torch.cuda.device_count() >= 1, 'CUDA is required for the formal bridge'


In [ ]:
# Mandatory static/CPU preflight before materializing the larger TinyStories suffix or spending GPU time.
compile_targets = [
    'src/minicells/clm_release_benchmark.py',
    'src/minicells/clm_release_reporting.py',
    'src/minicells/clm_release_visualization.py',
    'scripts/release/run_clm_0_3_release_bridge_worker.py',
    'scripts/release/run_clm_0_3_release_bridge_worker_entry.py',
    'scripts/release/run_clm_0_3_release_benchmark.py',
    'scripts/release/run_clm_0_3_release_benchmark_entry.py',
    'scripts/release/publish_clm_0_3_release_benchmark.py',
    'scripts/release/publish_clm_0_3_release_benchmark_entry.py',
]
subprocess.run([sys.executable, '-m', 'py_compile', *compile_targets], cwd=ROOT, check=True)
subprocess.run([
    sys.executable, '-m', 'pytest',
    'tests/research/03-routing-and-growth/test_clm_0_3_release_benchmark.py',
    'tests/research/03-routing-and-growth/test_clm_0_3_release_source.py',
    'tests/research/03-routing-and-growth/test_clm_probationary_mitosis.py',
    'tests/unit/routing_growth/test_growth_router.py',
    'tests/unit/routing_growth/test_growth_checkpoint.py',
    '-q',
], cwd=ROOT, check=True)


In [ ]:
RESULTS = ROOT / 'results/clm-0.3-release-benchmark'
RUN_FORMAL = False
RESTART_EXISTING = False  # set True only after a code-commit change; deletes old release-bridge evidence

cmd = [sys.executable, 'scripts/release/run_clm_0_3_release_benchmark_entry.py', '--output-root', str(RESULTS)]
if RESTART_EXISTING:
    cmd.append('--restart-existing')
if RUN_FORMAL:
    cmd.append('--execute')
subprocess.run(cmd, cwd=ROOT, check=True)


In [ ]:
# Inspect the machine-readable release recommendation.
import json
decision_path = RESULTS / 'decision.json'
if decision_path.exists():
    decision = json.loads(decision_path.read_text())
    print(json.dumps(decision, indent=2, sort_keys=True))
else:
    print('decision.json not produced yet')


In [ ]:
# Render the exact figures that will be published.
from IPython.display import Image, display, Markdown
for stem in [
    'figure-1-language-quality',
    'figure-2-machinery-bridge',
    'figure-3-developmental-selectivity',
    'figure-4-reference-cost',
]:
    path = RESULTS / 'figures' / f'{stem}.png'
    if path.exists():
        display(Markdown(f'### {stem}'))
        display(Image(filename=str(path)))
summary = RESULTS / 'PUBLIC-RELEASE-SUMMARY.md'
if summary.exists():
    display(Markdown(summary.read_text()))


In [ ]:
PUBLISH = False
if PUBLISH:
    subprocess.run([
        sys.executable,
        'scripts/release/publish_clm_0_3_release_benchmark_entry.py',
        '--push',
    ], cwd=ROOT, check=True)
